# Figure 2 — Vertical RMSE ratio

Pressure-dependent temperature and wind error ratios.


## 1. Load only the required common-grid data


In [ ]:
from pathlib import Path
import importlib
import sys
import numpy as np
import xarray as xr
import dask
from dask.distributed import Client, get_client

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from ml_implement_paper/ or its notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src import io as data_io
from src import metrics, plotting, preprocessing
plotting = importlib.reload(plotting)

config = data_io.load_config(PROJECT_ROOT / 'config' / 'paths.yaml')
analysis = config['analysis']

# ------------------------- User-adjustable settings -------------------------
OUTPUT_ROOT = Path('/global/cfs/cdirs/e3sm/www/zhan391/sea_crogs/online_diag')
ML_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/ml_method_2026')
REFERENCE_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/reference_nudge')
POST_SUBDIR = Path('post/atm/180x360_aave/ts/3hourly/1yr')
PERIOD = '201201_201212'
FORCE_COMPUTE = False  # True: overwrite selected case caches.
REQUIRED_VARIABLES = ['T200', 'T300', 'T400', 'T500', 'T600', 'U200', 'U850', 'V200', 'V850']
FIELD_LEVELS = {
    'T': {200: 'T200', 300: 'T300', 400: 'T400', 500: 'T500', 600: 'T600'},
    'U': {200: 'U200', 850: 'U850'},
    'V': {200: 'V200', 850: 'V850'},
}
paths = {
    'processed': OUTPUT_ROOT / 'processed',
    'figures': OUTPUT_ROOT / 'figures',
    'tables': OUTPUT_ROOT / 'tables',
}
for output_dir in paths.values():
    output_dir.mkdir(parents=True, exist_ok=True)
# Select cases here: keep CTRL and REF, and comment out any ML case you do not want.
CASE_DIRS = {
    'CTRL': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_CTRL',
    'UNET-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNET_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-IMT-C05': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR-IMT-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'UNETXTR-LCZ-C05A15-PTAP100': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_LANCZOS_NS6_SCL0.5_PTAPUNI100HPA_WQMADJ_WTVCON_PBL222_WVSMOOTH-S0.15',
    'REF': REFERENCE_ROOT / 'F20TR_ne30pg2_EC30to60E2r2_NDGUVTQ_IMT_3hr_pm-cpu_08-01-25',
}
FONT_SIZE = 14
FIGURE_SIZE = (9, 9)
FIGURE_LAYOUT = (2, 2)
LINE_WIDTH = 1.5
LEGEND_LOCATION = 'best'
LEGEND_FRAME = False
FIGURE_TITLE = 'Annual vertical RMSE ratio'
X_AXIS_LABEL = 'RMSE / CTRL RMSE'
Y_AXIS_LABEL = 'Pressure [hPa]'
REFERENCE_VALUE = 1.0
PRESSURE_SCALE = 'log'
COLORS = {
    'UNET-IMT': '#2878B5', 'UNETXTR-IMT': '#D95319',
    'UNETXTR-IMT-A15': '#9467BD', 'UNETXTR-IMT-C05': '#2CA02C',
    'UNETXTR-IMT-C05A15': '#8C564B', 'UNETXTR-LCZ-C05A15': '#17BECF',
    'UNETXTR-LCZ-C05A15-PTAP100': '#E377C2',
}
# ---------------------------------------------------------------------------

ML_CASES = tuple(name for name in CASE_DIRS if name not in {'CTRL', 'REF'})
if not {'CTRL', 'REF'}.issubset(CASE_DIRS):
    raise ValueError('CASE_DIRS must include CTRL and REF')
if not ML_CASES:
    raise ValueError('Select at least one ML case in CASE_DIRS')
ANALYSIS_CASES = ('CTRL', *ML_CASES)
DIAGNOSTIC_DIR = paths['processed'] / 'vertical_rmse_profiles'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_PATHS = {name: DIAGNOSTIC_DIR / f'{name}_{PERIOD}.nc' for name in ANALYSIS_CASES}

def cache_is_compatible(path, experiment):
    if not path.exists():
        return False
    try:
        with xr.open_dataset(path) as cached:
            variables = set(cached['variable'].values.astype(str))
            experiments = set(cached['experiment'].values.astype(str))
            return set(FIELD_LEVELS).issubset(variables) and experiment in experiments and 'rmse' in cached
    except (KeyError, OSError, ValueError):
        return False

CASES_TO_COMPUTE = tuple(
    name for name, path in DIAGNOSTIC_PATHS.items()
    if FORCE_COMPUTE or not cache_is_compatible(path, name)
)
RECOMPUTE = bool(CASES_TO_COMPUTE)
client = None
if RECOMPUTE:
    try:
        client = get_client()
    except ValueError:
        client = Client(n_workers=4, threads_per_worker=1, processes=False, dashboard_address=':0')

files = {
    name: [case_dir / POST_SUBDIR / f'{variable}_{PERIOD}.nc' for variable in REQUIRED_VARIABLES]
    for name, case_dir in CASE_DIRS.items() if name in ('REF', *CASES_TO_COMPUTE)
} if RECOMPUTE else {}
missing = {name: [str(path) for path in paths_ if not path.exists()] for name, paths_ in files.items()}
missing = {name: paths_ for name, paths_ in missing.items() if paths_}
if missing:
    details = '\n'.join(f'  {name}: {len(paths_)} missing file(s)' for name, paths_ in missing.items())
    raise FileNotFoundError('Postprocess the required variables first:\n' + details)

chunks = {'time': 32, 'lat': 45, 'lon': 90}
datasets = {
    name: xr.merge([xr.open_dataset(path, chunks=chunks, cache=False) for path in paths_], join='exact', compat='no_conflicts')
    for name, paths_ in files.items()
} if RECOMPUTE else None
datasets = {
    name: preprocessing.subset_time(ds, analysis['start_date'], analysis['end_date'])
    for name, ds in datasets.items()
} if RECOMPUTE else None
datasets = preprocessing.match_common_times(datasets) if RECOMPUTE else None
datasets = {name: preprocessing.daily_mean(ds) for name, ds in datasets.items()} if RECOMPUTE else None

if RECOMPUTE:
    sample = datasets[next(iter(datasets))][REQUIRED_VARIABLES[0]].isel(time=0, drop=True)
    area = np.cos(np.deg2rad(sample['lat'])).clip(min=0).broadcast_like(sample)
    area = area / area.sum()
    status = {'mode': 'compute', 'cases': CASES_TO_COMPUTE, 'datasets': {name: dict(ds.sizes) for name, ds in datasets.items()}}
else:
    status = {'mode': 'cached', 'paths': {name: str(path) for name, path in DIAGNOSTIC_PATHS.items()}}
status

## 2. Quality control


In [ ]:
# Run all finite-value checks together so Dask can share I/O efficiently.

qc_keys = []
qc_tasks = []
for case_name, dataset in datasets.items() if RECOMPUTE else []:
    for variable in REQUIRED_VARIABLES:
        qc_keys.append((case_name, variable))
        qc_tasks.extend([
            np.isfinite(dataset[variable]).any().data,
            (~np.isfinite(dataset[variable])).sum().data,
        ])
qc_values = dask.compute(*qc_tasks)
qc = {}
for index, key in enumerate(qc_keys):
    has_finite = bool(qc_values[2 * index])
    invalid_count = int(qc_values[2 * index + 1])
    if not has_finite:
        raise ValueError(f'{key[0]}:{key[1]} contains no finite values')
    qc.setdefault(key[0], {})[key[1]] = invalid_count
qc

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 164.76 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


## 3. Process and save the diagnostic data


In [ ]:
if RECOMPUTE:
    reference = datasets['REF']
    for name in CASES_TO_COMPUTE:
        profiles = []
        for family, level_fields in FIELD_LEVELS.items():
            by_level = [
                metrics.weighted_rmse(datasets[name][field], reference[field], area, list(area.dims)).mean('time')
                for field in level_fields.values()
            ]
            profiles.append(xr.concat(by_level, dim=xr.IndexVariable('plev', list(level_fields))))
        case_rmse = xr.concat(profiles, dim=xr.IndexVariable('variable', list(FIELD_LEVELS))).rename('rmse')
        case_diagnostic = xr.Dataset({'rmse': case_rmse}).expand_dims(experiment=[name])
        case_diagnostic.attrs.update({
            'experiment': name, 'period': PERIOD, 'reference_case': str(CASE_DIRS['REF']),
        })
        data_io.save_dataset(case_diagnostic, DIAGNOSTIC_PATHS[name])

rmse_product = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
ratios = xr.concat(
    [metrics.rmse_ratio(rmse_product['rmse'].sel(experiment=name), rmse_product['rmse'].sel(experiment='CTRL')) for name in ML_CASES],
    dim=xr.IndexVariable('experiment', np.asarray(ML_CASES, dtype=str)),
).rename('rmse_ratio')
diagnostic = xr.Dataset({'rmse': rmse_product['rmse'], 'rmse_ratio': ratios})
diagnostic

## 4. Reload the diagnostic product and create the figure


In [ ]:
FIGURE_PATH = paths['figures'] / 'fig02_vertical_rmse_ratio.png'
PLOT_VARIABLES = ['T', 'U', 'V']
PLOT_OPTIONS = {
    'figsize': FIGURE_SIZE, 'layout': FIGURE_LAYOUT,
    'colors': COLORS,
    'title': FIGURE_TITLE, 'xlabel': X_AXIS_LABEL,
    'ylabel': Y_AXIS_LABEL, 'reference_value': REFERENCE_VALUE, 'pressure_scale': PRESSURE_SCALE,
    'legend_loc': LEGEND_LOCATION, 'legend_frame': LEGEND_FRAME,
    'linewidth': LINE_WIDTH, 'font_size': FONT_SIZE,
}
rmse_product = xr.concat(
    [xr.open_dataset(DIAGNOSTIC_PATHS[name]) for name in ANALYSIS_CASES], dim='experiment'
)
ratios = xr.concat(
    [metrics.rmse_ratio(rmse_product['rmse'].sel(experiment=name), rmse_product['rmse'].sel(experiment='CTRL')) for name in ML_CASES],
    dim=xr.IndexVariable('experiment', np.asarray(ML_CASES, dtype=str)),
).rename('rmse_ratio')
diagnostic = xr.Dataset({'rmse': rmse_product['rmse'], 'rmse_ratio': ratios})
plotting.plot_vertical_rmse_ratio(diagnostic, PLOT_VARIABLES, FIGURE_PATH, **PLOT_OPTIONS)